In [5]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub

In [6]:
import os
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

In [7]:
df = pd.read_csv('cuidados_plantas.csv')
df.head()

,id,species_id,name,scientific_name,sunlight,watering,pruning
0,1,352,Ambrosia Apple,"[""Malus 'Ambrosia'""]",Ambrosia Apple (Malus 'Ambrosia') needs at lea...,Ambrosia Apple (Malus 'Ambrosia') plants requi...,The Ambrosia Apple (Malus 'Ambrosia') should b...
1,4,367,Honeycrisp Apple,"[""Malus 'Honeycrisp'""]","Sunlight is essential for all types of apples,...",Honeycrisp Apple (Malus 'Honeycrisp') should b...,Pruning Honeycrisp apples is an essential main...
2,5,1893,mandarin orange,"[""Citrus reticulata 'Clementine'""]",Mandarin orange plants should be pruned annual...,"Generally, mandarin orange trees need about on...",Mandarin oranges need plenty of sunlight to gr...
3,6,1895,navel orange,"[""Citrus sinensis 'Trovita'""]",Navel oranges need a considerable amount of su...,Water your navel orange tree (Citrus sinensis ...,Because Navel orange (Citrus sinensis 'Trovita...
4,7,1896,navel orange,"[""Citrus sinensis 'Washington'""]",The navel orange tree likes plenty of sunlight...,Navel oranges need to be watered 1–2 times per...,Naval orange trees should be pruned around the...


In [8]:
def row_to_text(row):
    return f"""### Instrucción:
Dame información sobre el cuidado de la planta {row['name']}.

### Respuesta:
La planta {row['name']} (nombre científico: {row['scientific_name']}) pertenece a la especie {row['species_id']}.
- Luz solar: {row['sunlight']}
- Riego: {row['watering']}
- Poda: {row['pruning']}"""

df['text'] = df.apply(row_to_text, axis=1)

In [9]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[['text']])
dataset = dataset.train_test_split(test_size=0.1)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [13]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = dataset["train"].map(tokenize, batched=True, remove_columns=["text"])
tokenized_eval = dataset["test"].map(tokenize, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir="./plantas-llm",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.157314,1.168945
2,1.164777,1.150394
3,1.139655,1.147521


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=1014, training_loss=1.188054601586547, metrics={'train_runtime': 7214.2979, 'train_samples_per_second': 1.123, 'train_steps_per_second': 0.141, 'total_flos': 8919103950028800.0, 'train_loss': 1.188054601586547})

In [14]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
repo_name = "plantas-llm-finetuned"

trainer.model.push_to_hub(repo_name, token=HF_TOKEN)
tokenizer.push_to_hub(repo_name, token=HF_TOKEN)

print(f"Modelo subido a: https://huggingface.co/{HfApi().whoami(token=HF_TOKEN)['name']}/{repo_name}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  55%|#####4    | 1.19MB / 2.18MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpk32r4s__/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Modelo subido a: https://huggingface.co/NewName4Me/plantas-llm-finetuned


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from huggingface_hub import HfApi

username = HfApi().whoami(token=HF_TOKEN)['name']
repo_id = f"{username}/plantas-llm-finetuned"

tokenizer = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
model_inf = AutoModelForCausalLM.from_pretrained(
    repo_id,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

tokenizer_config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/96 [00:00<?, ?it/s]

In [16]:
def preguntar(planta):
    prompt = f"""### Instrucción:
Dame información sobre el cuidado de la planta {planta}.

### Respuesta:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model_inf.device)

    with torch.no_grad():
        outputs = model_inf.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(respuesta.split("### Respuesta:")[-1].strip())

preguntar("Citrus berry")

The plant name is Citrus berry. The flower color of this plant is pink, and it has a fruit with an oval shape. It typically grows in full sun to partial shade and prefers warm climates. 

Plants should be kept healthy and watered regularly; they can also benefit from mulching around them. When pruning, prune back the branches that are dead or damaged, as well as those that appear crowded. If necessary, remove any flowers that have fallen off. Over time, the berries will mature and become ripe, so keep an eye on your plants and take care not to overprune. Pruning too much at once may cause damage to the tree's structure and make it more susceptible to disease. Also, if you notice any signs of pests or diseases, consider consulting a local arborist for help. Finally, don't forget to fertilize your citrus berry trees every year to ensure optimal growth!
